# CUDA-RL Baseline Evaluation (Kaggle Edition)

This notebook runs a **zero-shot baseline** experiment optimized for Kaggle kernels.
- Adjusts paths for `/kaggle/working/` and `/kaggle/input/` architectures.
- Loads reference kernels from KernelBench
- Prompts an LLM (Qwen2.5-Coder) with optimization guidance 
- Writes harness files, compiles, and collects Nsight metrics.
- Uses extensive step-by-step debugging prints.

## 0. Setup & Imports

In [1]:
import sys
import os
import json
import time
import textwrap
import traceback
import torch
from pathlib import Path
from datetime import datetime
import pandas as pd

# ─── Kaggle OFFLINE Path Configuration ───────────────────────────────────────
KAGGLE_INPUT = Path("/kaggle/input")

# Locate the custom script directory (contains script2_plan2.py v3)
try:
    SCRIPT2_DIR = next(KAGGLE_INPUT.rglob(
        "cuda-source/clean code_v10/clean code/hardware"
    ))
except StopIteration:
    print("[ERROR] Could not find hardware/ under /kaggle/input.")
    SCRIPT2_DIR = KAGGLE_INPUT

# Locate finetuning source directory (contains prompt.py)
try:
    RL_SRC_DIR = next(KAGGLE_INPUT.rglob(
        "cuda-source/clean code_v10/clean code/src/finetuning"
    ))
except StopIteration:
    print("[ERROR] Could not find src/finetuning/ under /kaggle/input.")
    RL_SRC_DIR = KAGGLE_INPUT

# Locate offline KernelBench dataset
try:
    DATASET_DIR = next(KAGGLE_INPUT.rglob(
        "datasets/mariahadjmessaoud/sakana-ai-dataset/dataset"
    ))
except StopIteration:
    DATASET_DIR = KAGGLE_INPUT / "sakana_ai_dataset"

# Locate offline Qwen model
try:
    MODEL_DIR = next(KAGGLE_INPUT.rglob("models/mariahadjmessaoud/qwen2-5-7b/pytorch/default/2/Qwen2.5-Coder-7B-Instruct/config.json")).parent

except StopIteration:
    try:
        MODEL_DIR = next(KAGGLE_INPUT.rglob("config.json")).parent
    except StopIteration:
        MODEL_DIR = KAGGLE_INPUT / "Qwen2.5-Coder-7B-Instruct"

RESULTS_DIR = Path("/kaggle/working/baseline_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SCRIPT2_DIR))
sys.path.insert(0, str(RL_SRC_DIR))

print("[SETUP] OFFLINE Paths configured securely on Kaggle.")
print(f"  Script2 dir : {SCRIPT2_DIR}")
print(f"  RL src dir  : {RL_SRC_DIR}")
print(f"  Dataset dir : {DATASET_DIR}")
print(f"  Model dir   : {MODEL_DIR}")
print(f"  Results dir : {RESULTS_DIR}")


[SETUP] OFFLINE Paths configured securely on Kaggle.
  Script2 dir : /kaggle/input/cuda-source/clean code_v10/clean code/hardware
  RL src dir  : /kaggle/input/cuda-source/clean code_v10/clean code/src/finetuning
  Dataset dir : /kaggle/input/datasets/mariahadjmessaoud/sakana-ai-dataset/dataset
  Model dir   : /kaggle/input/models/mariahadjmessaoud/qwen2-5-7b/pytorch/default/2/Qwen2.5-Coder-7B-Instruct/Qwen2.5-Coder-7B-Instruct
  Results dir : /kaggle/working/baseline_results


In [2]:
try:
    import script2_plan2 as harness
    harness.ensure_directories()
    print("[SETUP] script2_plan2 v4 imported successfully.")
    print(f"  Pending dir  : {harness.PENDING_DIR}")
    print(f"  Completed dir: {harness.COMPLETED_DIR}")
    print("  Pipeline     : load_inline  (no nvcc harness)")
    print("  Timing       : torch.cuda.Event")
    print("  Math check   : torch.allclose (atol=1e-3)")
    print("  Shape mode   : @SHAPES annotation → candidate fallback")
    print("  ncu profiling: Python driver (best-effort)")
except ImportError as e:
    print(f"[ERROR] Could not import script2_plan2: {e}")

[SETUP] script2_plan2 v4 imported successfully.
  Pending dir  : pending_kernels
  Completed dir: completed_results
  Pipeline     : load_inline  (no nvcc harness)
  Timing       : torch.cuda.Event
  Math check   : torch.allclose (atol=1e-3)
  Shape mode   : @SHAPES annotation → candidate fallback
  ncu profiling: Python driver (best-effort)


In [3]:
try:
    from datasets import load_from_disk
    print("[SETUP] datasets.load_from_disk imported successfully for OFFLINE loading.")
except ImportError as e:
    print(f"[ERROR] Could not import datasets module: {e}")

[SETUP] datasets.load_from_disk imported successfully for OFFLINE loading.


In [4]:
# ─── Config ──────────────────────────────────────────────────────────────────
N_KERNELS = 20
RANDOM_SEED = 42

print(f"[DATA] Loading dataset OFFLINE from {DATASET_DIR} ...")
try:
    level_1_path = DATASET_DIR / "level_1"
    if level_1_path.exists():
        dataset = load_from_disk(str(level_1_path))
    else:
        dataset = load_from_disk(str(DATASET_DIR))

    if 'level_1' in dataset:
        dataset = dataset['level_1']

    all_samples_raw = list(dataset)
    print(f"[DATA] Dataset loaded. {len(all_samples_raw)} total rows.")

    # FIX: deduplicate by Task_ID — the dataset has many rows per task.
    # all_samples_raw[:20] gives 20 copies of the SAME task (task_id=1).
    seen_tasks = set()
    unique_samples = []
    for s in all_samples_raw:
        tid = s.get('Task_ID', s.get('Op_Name', ''))
        if tid not in seen_tasks:
            seen_tasks.add(tid)
            unique_samples.append(s)

    sample = unique_samples[:N_KERNELS]
    print(f"[DATA] Unique tasks in dataset : {len(unique_samples)}")
    print(f"[DATA] Running on {len(sample)} unique tasks.")
    print(f"[DATA] Sample tasks: {[s.get('Op_Name','?') for s in sample[:5]]} ...")

except Exception as e:
    import traceback; traceback.print_exc()
    sample = []


[DATA] Loading dataset OFFLINE from /kaggle/input/datasets/mariahadjmessaoud/sakana-ai-dataset/dataset ...
[DATA] Dataset loaded. 12157 total rows.
[DATA] Unique tasks in dataset : 91
[DATA] Running on 20 unique tasks.
[DATA] Sample tasks: ['1_Square_matrix_multiplication_', '2_Standard_matrix_multiplication_', '3_Batched_matrix_multiplication', '4_Matrix_vector_multiplication_', '5_Matrix_scalar_multiplication'] ...


## 1. Dataset Exploration

Before running anything, we inspect the dataset to understand exactly what we are working with:
what schema it has, what the CUDA code looks like, and whether PyTorch extension patterns are present.

In [5]:
# ─── Load the dataset ─────────────────────────────────────────────────────────
print(f"[DATA] Loading dataset OFFLINE from {DATASET_DIR} ...")
try:
    level_1_path = DATASET_DIR / "level_1"
    if level_1_path.exists():
        dataset = load_from_disk(str(level_1_path))
    else:
        dataset = load_from_disk(str(DATASET_DIR))

    if 'level_1' in dataset:
        dataset = dataset['level_1']

    all_samples = list(dataset)
    print(f"[DATA] Dataset loaded. Total kernels available: {len(all_samples)}")
except Exception as e:
    print(f"[ERROR] Failed to load dataset: {e}")
    all_samples = []

[DATA] Loading dataset OFFLINE from /kaggle/input/datasets/mariahadjmessaoud/sakana-ai-dataset/dataset ...
[DATA] Dataset loaded. Total kernels available: 12157


In [6]:
# ─── Schema Exploration ───────────────────────────────────────────────────────
print("=== DATASET SCHEMA ===")
if all_samples:
    first = all_samples[0]
    print(f"Number of fields per entry : {len(first.keys())}")
    print(f"Field names                : {list(first.keys())}")
    print()
    for key, val in first.items():
        val_str = str(val)
        preview = val_str[:120] + '...' if len(val_str) > 120 else val_str
        print(f"  [{key}] ({type(val).__name__}) -> {preview}")

=== DATASET SCHEMA ===
Number of fields per entry : 19
Field names                : ['Op_Name', 'Level_ID', 'Task_ID', 'Kernel_Name', 'CUDA_Runtime', 'PyTorch_Native_Runtime', 'PyTorch_Compile_Runtime', 'CUDA_Speedup_Native', 'CUDA_Speedup_Compile', 'CUDA_Code', 'PyTorch_Code_Module', 'PyTorch_Code_Functional', 'Correct', 'Max_Diff', 'Error', 'NCU_Profile', 'Torch_Profile', 'Clang_Tidy', '__index_level_0__']

  [Op_Name] (str) -> 1_Square_matrix_multiplication_
  [Level_ID] (int) -> 1
  [Task_ID] (int) -> 1
  [Kernel_Name] (str) -> 1_Square_matrix_multiplication_
  [CUDA_Runtime] (float) -> 2.115
  [PyTorch_Native_Runtime] (float) -> 0.42108720541000366
  [PyTorch_Compile_Runtime] (float) -> 0.44516801834106445
  [CUDA_Speedup_Native] (float) -> 0.1990956053948007
  [CUDA_Speedup_Compile] (float) -> 0.2104813325489666
  [CUDA_Code] (str) -> #include <torch/extension.h>

#include <cuda.h>
#include <cuda_runtime.h>
#include <c10/cuda/CUDAException.h>

#define T...
  [PyTorch_Code_Module]

In [7]:
# ─── Operation Name Distribution ─────────────────────────────────────────────
print("=== ALL OPERATION NAMES IN DATASET ===")
op_names = [s.get('Op_Name', 'UNKNOWN') for s in all_samples]
op_series = pd.Series(op_names)
print(op_series.value_counts().to_string())
print(f"\nUnique operations: {op_series.nunique()} out of {len(all_samples)} total entries")

=== ALL OPERATION NAMES IN DATASET ===
42_Max_Pooling_2D                                                                                   521
98_KLDivLoss                                                                                        521
15_Matmul_for_lower_triangular_matrices                                                             501
8_Matmul_with_irregular_shapes_                                                                     121
5_Matrix_scalar_multiplication                                                                      121
6_Matmul_with_large_K_dimension_                                                                    121
7_Matmul_with_small_K_dimension_                                                                    121
10_3D_tensor_matrix_multiplication                                                                  121
9_Tall_skinny_matrix_multiplication_                                                                121
3_Batched_matrix_multipli

In [8]:
# ─── Inspect Reference CUDA Code for First 3 Entries ─────────────────────────
# This confirms the PyTorch extension pattern that causes the compiler mismatch.
print("=== REFERENCE CUDA CODE PREVIEW (first 3 entries) ===")
for i, s in enumerate(all_samples[:3]):
    code = s.get('CUDA_Code', '')
    print(f"\n--- Entry {i}: {s.get('Op_Name', 'UNKNOWN')} ---")
    print(f"  Code length: {len(code)} characters")
    # Check for PyTorch extension markers
    has_torch_ext    = '#include <torch/extension.h>' in code
    has_torch_tensor = 'torch::Tensor' in code
    has_at_dispatch  = 'AT_DISPATCH' in code
    has_global       = '__global__' in code
    print(f"  Contains #include <torch/extension.h> : {has_torch_ext}")
    print(f"  Contains torch::Tensor                : {has_torch_tensor}")
    print(f"  Contains AT_DISPATCH macro             : {has_at_dispatch}")
    print(f"  Contains __global__ kernel             : {has_global}")
    print(f"  First 600 chars of code:")
    print(textwrap.indent(code[:600], '    '))
    print()

=== REFERENCE CUDA CODE PREVIEW (first 3 entries) ===

--- Entry 0: 1_Square_matrix_multiplication_ ---
  Code length: 2579 characters
  Contains #include <torch/extension.h> : True
  Contains torch::Tensor                : True
  Contains AT_DISPATCH macro             : False
  Contains __global__ kernel             : True
  First 600 chars of code:
    #include <torch/extension.h>

    #include <cuda.h>
    #include <cuda_runtime.h>
    #include <c10/cuda/CUDAException.h>

    #define TILE_SIZE 16

    #define CHECK_CUDA(x) TORCH_CHECK(x.is_cuda(), #x " must be a CUDA tensor")
    #define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
    #define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)
    #define CHECK_FLOAT(x) TORCH_CHECK(x.scalar_type() == torch::kFloat32, #x " must be a float32 tensor")

    __global__ void matmul_tiled_kernel(const float* __restrict__ A, const float* __restrict__ B, float* __restrict__ C, int N) {
        __shared__ flo




In [9]:
# ─── Dataset-wide Pattern Statistics ─────────────────────────────────────────
print("=== DATASET-WIDE PATTERN STATISTICS ===")
stats = {
    'has_torch_extension_h': 0,
    'has_torch_tensor':      0,
    'has_at_dispatch':       0,
    'has_global_kernel':     0,
    'code_length_chars':     [],
}
for s in all_samples:
    # Use 'or ""' to catch explicit None values alongside missing keys
    code = s.get('CUDA_Code') or ''
    if '#include <torch/extension.h>' in code: stats['has_torch_extension_h'] += 1
    if 'torch::Tensor' in code:                stats['has_torch_tensor'] += 1
    if 'AT_DISPATCH' in code:                  stats['has_at_dispatch'] += 1
    if '__global__' in code:                   stats['has_global_kernel'] += 1
    stats['code_length_chars'].append(len(code))

lengths = pd.Series(stats['code_length_chars'])
total = len(all_samples)
print(f"Total entries in dataset      : {total}")
print(f"Has torch/extension.h         : {stats['has_torch_extension_h']} / {total} ({100*stats['has_torch_extension_h']/total:.0f}%)")
print(f"Has torch::Tensor             : {stats['has_torch_tensor']} / {total} ({100*stats['has_torch_tensor']/total:.0f}%)")
print(f"Has AT_DISPATCH macro         : {stats['has_at_dispatch']} / {total} ({100*stats['has_at_dispatch']/total:.0f}%)")
print(f"Has __global__ kernel         : {stats['has_global_kernel']} / {total} ({100*stats['has_global_kernel']/total:.0f}%)")
print(f"\nCode length stats (chars):")
print(f"  Min    : {lengths.min():.0f}")
print(f"  Median : {lengths.median():.0f}")
print(f"  Mean   : {lengths.mean():.0f}")
print(f"  Max    : {lengths.max():.0f}")
print()
print("NOTE: Since ~100% of the dataset uses torch/extension.h, the LLM WILL")
print("      generate PyTorch extension code. script2_plan2.py must compile with")
print("      torch include/library paths — which is exactly what v2 now does.")

=== DATASET-WIDE PATTERN STATISTICS ===
Total entries in dataset      : 12157
Has torch/extension.h         : 11240 / 12157 (92%)
Has torch::Tensor             : 9695 / 12157 (80%)
Has AT_DISPATCH macro         : 3992 / 12157 (33%)
Has __global__ kernel         : 11166 / 12157 (92%)

Code length stats (chars):
  Min    : 0
  Median : 3432
  Mean   : 3548
  Max    : 30630

NOTE: Since ~100% of the dataset uses torch/extension.h, the LLM WILL
      generate PyTorch extension code. script2_plan2.py must compile with
      torch include/library paths — which is exactly what v2 now does.


In [10]:
# ─── Display the 20 unique selected tasks ────────────────────────────────────
print(f"[DATA] Selected {len(sample)} kernels.")
for idx, s in enumerate(sample):
    print(f"  [{idx:02d}] Task_ID={s.get('Task_ID','?')} | {s.get('Op_Name','UNKNOWN')}")


[DATA] Selected 20 kernels.
  [00] Task_ID=1 | 1_Square_matrix_multiplication_
  [01] Task_ID=2 | 2_Standard_matrix_multiplication_
  [02] Task_ID=3 | 3_Batched_matrix_multiplication
  [03] Task_ID=4 | 4_Matrix_vector_multiplication_
  [04] Task_ID=5 | 5_Matrix_scalar_multiplication
  [05] Task_ID=6 | 6_Matmul_with_large_K_dimension_
  [06] Task_ID=7 | 7_Matmul_with_small_K_dimension_
  [07] Task_ID=8 | 8_Matmul_with_irregular_shapes_
  [08] Task_ID=9 | 9_Tall_skinny_matrix_multiplication_
  [09] Task_ID=10 | 10_3D_tensor_matrix_multiplication
  [10] Task_ID=11 | 11_4D_tensor_matrix_multiplication
  [11] Task_ID=12 | 12_Matmul_with_diagonal_matrices_
  [12] Task_ID=13 | 13_Matmul_for_symmetric_matrices
  [13] Task_ID=14 | 14_Matmul_for_upper_triangular_matrices
  [14] Task_ID=15 | 15_Matmul_for_lower_triangular_matrices
  [15] Task_ID=16 | 16_Matmul_with_transposed_A
  [16] Task_ID=17 | 17_Matmul_with_transposed_B
  [17] Task_ID=18 | 18_Matmul_with_transposed_both
  [18] Task_ID=19 | 1

## 2. LLM Client
Loads the model. Kaggle provides an ephemereal disk, so we just load it directly to memory.

In [11]:
import struct
from pathlib import Path

model_dir = Path(MODEL_DIR)
shards = sorted(model_dir.glob("*.safetensors"))

print(f"Found {len(shards)} shard files:\n")
for shard in shards:
    size_gb = shard.stat().st_size / (1024**3)
    # Read the first 8 bytes — safetensor header length prefix
    try:
        with open(shard, 'rb') as f:
            first8 = f.read(8)
        if len(first8) < 8:
            status = "CORRUPT — file too small to have a valid header"
        else:
            header_len = struct.unpack('<Q', first8)[0]
            if header_len == 0 or header_len > 100_000_000:
                status = f"CORRUPT — invalid header length: {header_len}"
            else:
                status = f"OK — header length: {header_len} bytes"
    except Exception as e:
        status = f"ERROR — {e}"
    
    print(f"  {shard.name:45s} {size_gb:.2f} GB  →  {status}")

Found 4 shard files:

  model-00001-of-00004.safetensors              4.54 GB  →  OK — header length: 11872 bytes
  model-00002-of-00004.safetensors              4.59 GB  →  OK — header length: 13976 bytes
  model-00003-of-00004.safetensors              4.03 GB  →  OK — header length: 12840 bytes
  model-00004-of-00004.safetensors              1.02 GB  →  OK — header length: 120 bytes


In [12]:
import os
import sys
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

KAGGLE_INPUT = Path("/kaggle/input")

try:
    MODEL_DIR = next(KAGGLE_INPUT.rglob("Qwen2.5-Coder-7B-Instruct/config.json")).parent
except StopIteration:
    try:
        MODEL_DIR = next(KAGGLE_INPUT.rglob("config.json")).parent
    except StopIteration:
        raise RuntimeError("[ERROR] Could not locate model config.json under /kaggle/input.")

print(f"[MODEL] Resolved MODEL_DIR = {MODEL_DIR}")
assert (MODEL_DIR / "config.json").exists(), "config.json not found in MODEL_DIR"

print(f"[MODEL] Loading tokenizer from {MODEL_DIR} ...")
tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
)
print(f"[MODEL] Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

print(f"[MODEL] Loading model weights from {MODEL_DIR} ...")
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    device_map="auto",
    torch_dtype=torch.bfloat16,
    local_files_only=True,
)
model.eval()
print(f"[MODEL] Model loaded on: {next(model.parameters()).device}")

def generate_optimized_kernel(system_prompt: str, user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=4096,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_ids = [
        output[len(inp):]
        for inp, output in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("[MODEL] Ready. generate_optimized_kernel() is available.")

[MODEL] Resolved MODEL_DIR = /kaggle/input/models/mariahadjmessaoud/qwen2-5-7b/pytorch/default/2/Qwen2.5-Coder-7B-Instruct/Qwen2.5-Coder-7B-Instruct
[MODEL] Loading tokenizer from /kaggle/input/models/mariahadjmessaoud/qwen2-5-7b/pytorch/default/2/Qwen2.5-Coder-7B-Instruct/Qwen2.5-Coder-7B-Instruct ...


`torch_dtype` is deprecated! Use `dtype` instead!


[MODEL] Tokenizer loaded. Vocab size: 151643
[MODEL] Loading model weights from /kaggle/input/models/mariahadjmessaoud/qwen2-5-7b/pytorch/default/2/Qwen2.5-Coder-7B-Instruct/Qwen2.5-Coder-7B-Instruct ...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[MODEL] Model loaded on: cuda:0
[MODEL] Ready. generate_optimized_kernel() is available.


In [13]:
import torch

print("=== GPU Status Check ===")
print(f"Is CUDA available? {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_id = torch.cuda.current_device()
    print(f"Current Active Device ID: cuda:{device_id}")
    print(f"GPU Name: {torch.cuda.get_device_name(device_id)}")

    allocated = torch.cuda.memory_allocated(device_id) / (1024**3)
    reserved  = torch.cuda.memory_reserved(device_id)  / (1024**3)
    total     = torch.cuda.get_device_properties(device_id).total_memory / (1024**3)

    print(f"Total VRAM                    : {total:.2f} GB")
    print(f"VRAM Allocated (model in use) : {allocated:.2f} GB")
    print(f"VRAM Reserved  (PyTorch cache): {reserved:.2f} GB")
    print(f"VRAM Free (estimated)         : {total - reserved:.2f} GB")
else:
    print("Warning: PyTorch is currently restricted to CPU!")

=== GPU Status Check ===
Is CUDA available? True
Current Active Device ID: cuda:0
GPU Name: NVIDIA RTX PRO 6000 Blackwell Server Edition
Total VRAM                    : 94.97 GB
VRAM Allocated (model in use) : 14.19 GB
VRAM Reserved  (PyTorch cache): 14.21 GB
VRAM Free (estimated)         : 80.77 GB


## 3. Prompt Management

In [14]:
try:
    from prompt import CUDA_SYSTEM_PROMPT as SYSTEM_PROMPT
    from prompt import build_user_prompt
    print("[PROMPT] System and User prompts loaded successfully from finetuning.prompt.")
except ImportError as e:
    print(f"[ERROR] Could not import prompts: {e}")

[PROMPT] System and User prompts loaded successfully from finetuning.prompt.


## 4. Harness Writer

In [15]:
import re
from pathlib import Path

def _extract_cuda_code(llm_output: str) -> str:
    """
    Strip markdown code fences from raw LLM output.

    ROOT CAUSE OF ALL COMPILATION_ERRORs (v1, v2, v3):
    The LLM wraps its answer in markdown:

        ```cuda
        #include <torch/extension.h>
        ...
        ```

    Written raw to .cu, nvcc sees the backtick on line 4:
        error: unrecognized token
          ```cuda
          ^
    This corrupts the C++ parser state, producing 50+ cascade errors
    across valid system headers (stl_stack.h, autograd.h, etc.)
    that are all meaningless — caused by this one unstripped fence.
    """
    fence_re = re.compile(
        r'```(?:cuda|cpp|c\+\+|c|cu)?\s*\n(.*?)```',
        re.DOTALL | re.IGNORECASE
    )
    matches = fence_re.findall(llm_output)
    if matches:
        return "\n\n".join(m.strip() for m in matches)
    stripped = llm_output.strip()
    if stripped.startswith('#') or stripped.startswith('//') or stripped.startswith('/*'):
        return stripped
    return stripped


def write_kernel(trial_id: str, task_name: str, llm_output: str) -> Path:
    """Write cleaned CUDA source to pending_kernels/<trial_id>_kernel.cu."""
    cuda_src = _extract_cuda_code(llm_output)
    path = Path(harness.PENDING_DIR) / f"{trial_id}_kernel.cu"
    path.write_text(cuda_src, encoding='utf-8')
    return path


print("[KERNEL WRITER] write_kernel() ready. Fence stripping: ENABLED.")


[KERNEL WRITER] write_kernel() ready. Fence stripping: ENABLED.


In [16]:
# ─── Fence stripper sanity check ────────────────────────────────────────────
_MOCK = """
Here is the optimized kernel:

```cuda
#include <torch/extension.h>
#include <cuda_runtime.h>

torch::Tensor baseline_forward(torch::Tensor x) { return x.clone(); }
torch::Tensor optimized_forward(torch::Tensor x) { return x.clone(); }

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("baseline_forward",  &baseline_forward);
    m.def("optimized_forward", &optimized_forward);
}
```
"""
out = _extract_cuda_code(_MOCK)
print(f"Input : {len(_MOCK)} chars, starts with: {_MOCK.strip()[:30]!r}")
print(f"Output: {len(out)} chars, starts with: {out[:40]!r}")
assert '```' not in out,   "FAIL: backticks still in output!"
assert '#include' in out,  "FAIL: no #include found!"
print("PASSED — fence stripper is working correctly.")


Input : 390 chars, starts with: 'Here is the optimized kernel:\n'
Output: 345 chars, starts with: '#include <torch/extension.h>\n#include <c'
PASSED — fence stripper is working correctly.


## 5. Explicitly Debugged Evaluation Loop

In [17]:
results = []

print("\n=======================================================")
print("[EVAL] Starting Evaluation Loop  (script2 v4 / shape-aware pipeline)")
print("=======================================================\n")


for i, entry in enumerate(sample):
    trial_id  = f"baseline_{i:03d}"
    task_name = entry.get("Op_Name", f"Task_{i}")
    ref_src   = entry.get("CUDA_Code", "")

    print(f"\n[LOOP] ---> Iteration {i+1}/{len(sample)}: {task_name}")
    sys.stdout.flush()

    # ── Step 1: LLM generation ────────────────────────────────────────────────
    print("       [Step 1] Prompting LLM …")
    llm_output = None
    llm_time   = None
    try:
        user_prompt = build_user_prompt(task_name, ref_src)
        t0          = time.time()
        llm_output  = generate_optimized_kernel(SYSTEM_PROMPT, user_prompt)
        llm_time    = time.time() - t0
        print(f"       [Step 1] SUCCESS  — {llm_time:.1f}s, {len(llm_output)} chars")
    except Exception as e:
        err_str = str(e)
        if "illegal memory access" in err_str or "cudaError" in err_str:
            print(f"       [Step 1] FATAL: GPU context corrupted. Stopping loop.")
            break
        print(f"       [Step 1] FAILED: {e}")
        results.append({
            "trial_id":        trial_id,
            "task_name":       task_name,
            "status":          "LLM_ERROR",
            "reward":          -1.0,
            "llm_time_s":      None,
            "shapes_declared": False,
        })
        continue
    sys.stdout.flush()

    # ── Step 2: Parse @SHAPES annotation before writing ──────────────────────
    # This is purely for logging/observability — script2 does its own parse
    # internally. We log it here so you can see coverage per trial.
    print("       [Step 2] Parsing @SHAPES annotation …")
    parsed_shapes = harness.parse_shapes_from_source(llm_output)
    if parsed_shapes:
        print(f"       [Step 2] @SHAPES found: {parsed_shapes}")
    else:
        print("       [Step 2] WARNING — no @SHAPES annotation. Candidate fallback will be used.")
    sys.stdout.flush()

    # ── Step 3: Write raw LLM source to _kernel.cu ───────────────────────────
    print("       [Step 3] Writing _kernel.cu …")
    try:
        kernel_path = write_kernel(trial_id, task_name, llm_output)
        print(f"       [Step 3] SUCCESS  — {kernel_path}")
    except Exception as e:
        print(f"       [Step 3] FAILED: {e}")
        traceback.print_exc()
        results.append({
            "trial_id":        trial_id,
            "task_name":       task_name,
            "status":          "WRITE_ERROR",
            "reward":          -1.0,
            "llm_time_s":      round(llm_time, 2) if llm_time else None,
            "shapes_declared": bool(parsed_shapes),
        })
        continue
    sys.stdout.flush()

    # ── Step 4: script2 v4 pipeline ──────────────────────────────────────────
    # process_trial() reads the .cu file itself — no need to pass cuda_src here.
    # Shape parsing happens inside _benchmark_subprocess via the saved .cu file.
    print("       [Step 4] Running script2 v4 pipeline (compile → benchmark → profile) …")
    try:
        harness.process_trial(trial_id)
        print("       [Step 4] Pipeline returned control.")
    except Exception as e:
        print(f"       [Step 4] Exception: {e}")
        traceback.print_exc()
    sys.stdout.flush()

    # ── Step 5: Parse JSON result ─────────────────────────────────────────────
    print("       [Step 5] Reading result JSON …")
    result_path    = Path(harness.COMPLETED_DIR) / f"{trial_id}_results.json"
    result_payload = {"status": "PIPELINE_ERROR", "reward": -1.0, "metrics": {}}

    if result_path.exists():
        try:
            with open(result_path, "r") as f:
                result_payload = json.load(f)
            print("       [Step 5] SUCCESS")
        except Exception as e:
            print(f"       [Step 5] Corrupted JSON: {e}")
    else:
        print(f"       [Step 5] WARNING — result JSON not found at {result_path}")
    sys.stdout.flush()

    # ── Step 6: Record ────────────────────────────────────────────────────────
    status  = result_payload.get("status",  "UNKNOWN")
    reward  = result_payload.get("reward",  -1.0)
    metrics = result_payload.get("metrics", {}) or {}

    print(f"       [RESULT] Status={status}  Reward={reward:.4f}")
    if metrics.get("baseline_time_ns") and metrics.get("optimized_time_ns"):
        b   = metrics["baseline_time_ns"]
        o   = max(metrics["optimized_time_ns"], 1)
        spd = b / o
        print(
            f"       [RESULT] Speedup={spd:.3f}x | "
            f"Occ={metrics.get('occupancy_pct')}% | "
            f"Comp={metrics.get('compute_pct')}% | "
            f"Mem={metrics.get('memory_pct')}%"
        )

    results.append({
        "trial_id":          trial_id,
        "task_name":         task_name,
        "status":            status,
        "reward":            reward,
        "llm_time_s":        round(llm_time, 2) if llm_time else None,
        "shapes_declared":   bool(parsed_shapes),
        "baseline_time_ns":  metrics.get("baseline_time_ns"),
        "optimized_time_ns": metrics.get("optimized_time_ns"),
        "occupancy_pct":     metrics.get("occupancy_pct"),
        "compute_pct":       metrics.get("compute_pct"),
        "memory_pct":        metrics.get("memory_pct"),
    })

    try:
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    except Exception:
        pass

    print(f"[LOOP] <--- Finished iteration {i+1}")
    sys.stdout.flush()

print("\n=======================================================")
print("[EVAL] Evaluation loop complete.")
print("=======================================================")


[EVAL] Starting Evaluation Loop  (script2 v4 / shape-aware pipeline)


[LOOP] ---> Iteration 1/20: 1_Square_matrix_multiplication_
       [Step 1] Prompting LLM …
       [Step 1] SUCCESS  — 23.2s, 4931 chars
       [Step 2] Parsing @SHAPES annotation …
       [Step 2] @SHAPES found: [{'dtype': 'torch.float32', 'shape': [1024, 1024]}, {'dtype': 'torch.float32', 'shape': [1024, 1024]}]
       [Step 3] Writing _kernel.cu …
       [Step 3] SUCCESS  — pending_kernels/baseline_000_kernel.cu
       [Step 4] Running script2 v4 pipeline (compile → benchmark → profile) …
[2026-05-02 16:36:16] [INFO] Processing trial: baseline_000
[2026-05-02 16:36:16] [INFO] baseline_000 | Compiling via load_inline …
[1/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output cuda.cuda.o.d -DTORCH_EXTENSION_NAME=kernel_baseline_000 -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/incl

## 6. Save & Summarize Results

In [18]:
df = pd.DataFrame(results)

ts         = datetime.utcnow().strftime("%Y%m%d_%H%M")
csv_path   = RESULTS_DIR / f"baseline_{ts}.csv"
jsonl_path = RESULTS_DIR / f"baseline_metrics_{ts}.jsonl"

df.to_csv(csv_path, index=False)
df.to_json(jsonl_path, orient="records", lines=True)

print(f"[SUMMARY] Results saved:")
print(f"  - CSV  : {csv_path}")
print(f"  - JSONL: {jsonl_path}")

[SUMMARY] Results saved:
  - CSV  : /kaggle/working/baseline_results/baseline_20260502_1649.csv
  - JSONL: /kaggle/working/baseline_results/baseline_metrics_20260502_1649.jsonl


/tmp/ipykernel_163/2812503115.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts         = datetime.utcnow().strftime("%Y%m%d_%H%M")


## 7. Results Report & Visualizations

In [19]:
import matplotlib
matplotlib.use('Agg')   # Safe for Kaggle — no display server needed
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

PLOT_DIR = RESULTS_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Color palette by status ─────────────────────────────────────────────────
STATUS_COLORS = {
    'SUCCESS':           '#2ecc71',
    'WRONG_MATH':        '#f39c12',
    'COMPILATION_ERROR': '#e74c3c',
    'RUNTIME_ERROR':     '#c0392b',
    'TIMEOUT':           '#9b59b6',
    'SYSTEM_ERROR':      '#e67e22',
    'LLM_ERROR':         '#7f8c8d',
    'PIPELINE_ERROR':    '#34495e',
    'UNKNOWN':           '#bdc3c7',
}

def status_color(s):
    return STATUS_COLORS.get(s, '#bdc3c7')

print("[PLOT] Matplotlib ready. All plots saved to:", PLOT_DIR)

[PLOT] Matplotlib ready. All plots saved to: /kaggle/working/baseline_results/plots


In [20]:
# ─── Plot 1: Status Distribution Pie Chart ───────────────────────────────────
status_counts = df['status'].value_counts()

fig, ax = plt.subplots(figsize=(7, 7))
colors  = [status_color(s) for s in status_counts.index]
wedges, texts, autotexts = ax.pie(
    status_counts.values,
    labels=status_counts.index,
    autopct='%1.0f%%',
    colors=colors,
    startangle=140,
    textprops={'fontsize': 11},
)
for at in autotexts:
    at.set_fontweight('bold')
ax.set_title(f'Trial Status Distribution\n({len(df)} total trials)', fontsize=13, fontweight='bold')
plt.tight_layout()
p1 = PLOT_DIR / 'status_distribution.png'
plt.savefig(p1, dpi=150, bbox_inches='tight')
plt.show()
print(f"[PLOT] Saved: {p1}")

[PLOT] Saved: /kaggle/working/baseline_results/plots/status_distribution.png


In [21]:
# ─── Plot 2: Reward per Trial (bar chart) ────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
bar_colors = [status_color(s) for s in df['status']]
bars = ax.bar(df['trial_id'], df['reward'], color=bar_colors, edgecolor='black', linewidth=0.4)

# Reference lines
ax.axhline(y=1.0, color='navy',  linestyle='--', linewidth=1.2, label='Reward = 1.0 (no change)')
ax.axhline(y=0.0, color='grey',  linestyle=':',  linewidth=1.0, label='Reward = 0.0 (wrong math)')
ax.axhline(y=-1.0, color='red',  linestyle=':',  linewidth=1.0, label='Reward = -1.0 (hard failure)')

ax.set_xlabel('Trial ID', fontsize=11)
ax.set_ylabel('Reward (baseline_time / optimized_time)', fontsize=11)
ax.set_title('Reward per Trial', fontsize=13, fontweight='bold')
ax.set_xticks(range(len(df)))
ax.set_xticklabels(df['trial_id'], rotation=45, ha='right', fontsize=8)

# Legend for statuses
patches = [mpatches.Patch(color=v, label=k) for k, v in STATUS_COLORS.items() if k in df['status'].values]
ax.legend(handles=patches, loc='upper right', fontsize=8)

plt.tight_layout()
p2 = PLOT_DIR / 'reward_per_trial.png'
plt.savefig(p2, dpi=150, bbox_inches='tight')
plt.show()
print(f"[PLOT] Saved: {p2}")

[PLOT] Saved: /kaggle/working/baseline_results/plots/reward_per_trial.png


In [22]:
# ─── Plot 3: Speedup for successful trials ────────────────────────────────────
success_df = df[df['status'] == 'SUCCESS'].copy()

if len(success_df) > 0:
    success_df['speedup'] = success_df['baseline_time_ns'] / success_df['optimized_time_ns']

    fig, ax = plt.subplots(figsize=(max(8, len(success_df) * 0.8), 5))
    bar_cols = ['#2ecc71' if s >= 1.0 else '#e74c3c' for s in success_df['speedup']]
    ax.bar(success_df['trial_id'], success_df['speedup'], color=bar_cols, edgecolor='black', linewidth=0.5)
    ax.axhline(y=1.0, color='navy', linestyle='--', linewidth=1.2, label='Speedup = 1.0 (baseline)')
    ax.set_xlabel('Trial ID', fontsize=11)
    ax.set_ylabel('Speedup (x)', fontsize=11)
    ax.set_title(f'Kernel Speedup for Successful Trials (n={len(success_df)})', fontsize=13, fontweight='bold')
    ax.set_xticks(range(len(success_df)))
    ax.set_xticklabels(success_df['trial_id'], rotation=45, ha='right', fontsize=9)
    ax.legend(fontsize=9)
    plt.tight_layout()
    p3 = PLOT_DIR / 'speedup_successful.png'
    plt.savefig(p3, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[PLOT] Saved: {p3}")
else:
    print("[PLOT] No successful trials — speedup plot skipped.")

[PLOT] Saved: /kaggle/working/baseline_results/plots/speedup_successful.png


In [23]:
# ─── Plot 4: GPU Utilization Metrics for Successful Trials ───────────────────
if len(success_df) > 0:
    metric_cols  = ['occupancy_pct', 'compute_pct', 'memory_pct']
    metric_labels = ['Occupancy (%)', 'Compute Throughput (%)', 'Memory Throughput (%)']
    metric_colors = ['#3498db', '#e67e22', '#9b59b6']

    available = [c for c in metric_cols if c in success_df.columns and success_df[c].notna().any()]

    if available:
        x = np.arange(len(success_df))
        width = 0.25
        fig, ax = plt.subplots(figsize=(max(10, len(success_df)), 5))

        for j, col in enumerate(available):
            label = metric_labels[metric_cols.index(col)]
            color = metric_colors[metric_cols.index(col)]
            vals  = success_df[col].fillna(0).values
            ax.bar(x + j * width, vals, width, label=label, color=color, alpha=0.85, edgecolor='black', linewidth=0.4)

        ax.set_xlabel('Trial ID', fontsize=11)
        ax.set_ylabel('Percentage (%)', fontsize=11)
        ax.set_title('GPU Utilization Metrics — Optimized Kernel (Successful Trials)', fontsize=13, fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(success_df['trial_id'].values, rotation=45, ha='right', fontsize=9)
        ax.set_ylim(0, 110)
        ax.legend(fontsize=9)
        plt.tight_layout()
        p4 = PLOT_DIR / 'gpu_utilization.png'
        plt.savefig(p4, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"[PLOT] Saved: {p4}")
    else:
        print("[PLOT] No utilization metric columns available — skipped.")
else:
    print("[PLOT] No successful trials — utilization plot skipped.")

[PLOT] No utilization metric columns available — skipped.


In [24]:
# ─── Plot 5: LLM Generation Time per Trial ───────────────────────────────────
if 'llm_time_s' in df.columns and df['llm_time_s'].notna().any():
    fig, ax = plt.subplots(figsize=(14, 4))
    llm_colors = [status_color(s) for s in df['status']]
    ax.bar(df['trial_id'], df['llm_time_s'].fillna(0), color=llm_colors, edgecolor='black', linewidth=0.4)
    ax.set_xlabel('Trial ID', fontsize=11)
    ax.set_ylabel('LLM Generation Time (s)', fontsize=11)
    ax.set_title('LLM Generation Time per Trial', fontsize=13, fontweight='bold')
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df['trial_id'], rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    p5 = PLOT_DIR / 'llm_generation_time.png'
    plt.savefig(p5, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[PLOT] Saved: {p5}")

[PLOT] Saved: /kaggle/working/baseline_results/plots/llm_generation_time.png


In [25]:
# ─── Final Text Report ────────────────────────────────────────────────────────
print("\n" + "="*60)
print("         BASELINE EXPERIMENT — FULL RESULTS REPORT")
print("="*60)

total   = len(df)
success = df[df['status'] == 'SUCCESS']
n_ok    = len(success)

print(f"\n[OVERVIEW]")
print(f"  Total trials run     : {total}")
print(f"  Successful trials    : {n_ok} ({100*n_ok/total:.0f}%)")
print(f"  Failed trials        : {total - n_ok} ({100*(total-n_ok)/total:.0f}%)")

print(f"\n[STATUS BREAKDOWN]")
for status_val, count in df['status'].value_counts().items():
    print(f"  {status_val:<22}: {count:>3}  ({100*count/total:.0f}%)")

if n_ok > 0:
    speedups = success['baseline_time_ns'] / success['optimized_time_ns']
    print(f"\n[SPEEDUP STATISTICS — Successful Trials Only]")
    print(f"  Mean speedup   : {speedups.mean():.3f}x")
    print(f"  Median speedup : {speedups.median():.3f}x")
    print(f"  Best speedup   : {speedups.max():.3f}x  (task: {success.loc[speedups.idxmax(), 'task_name']})")
    print(f"  Worst speedup  : {speedups.min():.3f}x  (task: {success.loc[speedups.idxmin(), 'task_name']})")
    improved = (speedups > 1.0).sum()
    print(f"  Kernels faster than baseline  : {improved}/{n_ok} ({100*improved/n_ok:.0f}%)")
    regressed = (speedups < 1.0).sum()
    print(f"  Kernels slower than baseline  : {regressed}/{n_ok} ({100*regressed/n_ok:.0f}%)")

    occ_vals  = success['occupancy_pct'].dropna()  if 'occupancy_pct'  in success.columns else pd.Series()
    comp_vals = success['compute_pct'].dropna()     if 'compute_pct'    in success.columns else pd.Series()
    mem_vals  = success['memory_pct'].dropna()      if 'memory_pct'     in success.columns else pd.Series()

    if len(occ_vals): print(f"\n[GPU UTILIZATION — Optimized Kernels]")
    if len(occ_vals):  print(f"  Mean Occupancy         : {occ_vals.mean():.1f}%")
    if len(comp_vals): print(f"  Mean Compute Throughput: {comp_vals.mean():.1f}%")
    if len(mem_vals):  print(f"  Mean Memory Throughput : {mem_vals.mean():.1f}%")

if 'llm_time_s' in df.columns and df['llm_time_s'].notna().any():
    print(f"\n[LLM PERFORMANCE]")
    print(f"  Mean generation time   : {df['llm_time_s'].mean():.1f}s")
    print(f"  Total generation time  : {df['llm_time_s'].sum():.1f}s")

print(f"\n[FULL TRIAL TABLE]")
display_cols = ['trial_id', 'task_name', 'status', 'reward', 'llm_time_s']
if 'occupancy_pct' in df.columns: display_cols += ['occupancy_pct', 'compute_pct', 'memory_pct']
print(df[display_cols].to_string(index=False))

print("\n" + "="*60)
print("[DIAGNOSIS]")
comp_err = (df['status'] == 'COMPILATION_ERROR').sum()
if comp_err > 0:
    print(f"  {comp_err} COMPILATION_ERROR(s) detected.")
    print("  Check that script2_plan2.py v2 is being used (with build_nvcc_compile_cmd).")
    print("  Verify torch.utils.cpp_extension.include_paths() returns valid paths on this machine.")
wrong_math = (df['status'] == 'WRONG_MATH').sum()
if wrong_math > 0:
    print(f"  {wrong_math} WRONG_MATH result(s): the LLM generated incorrect optimizations.")
    print("  Consider tightening the system prompt or increasing temperature to resample.")
if n_ok == total:
    print("  All trials succeeded! Review speedup distribution to identify best/worst kernels.")
print("="*60)


         BASELINE EXPERIMENT — FULL RESULTS REPORT

[OVERVIEW]
  Total trials run     : 20
  Successful trials    : 6 (30%)
  Failed trials        : 14 (70%)

[STATUS BREAKDOWN]
  SUCCESS               :   6  (30%)
  COMPILATION_ERROR     :   5  (25%)
  RUNTIME_ERROR         :   5  (25%)
  WRONG_MATH            :   4  (20%)

[SPEEDUP STATISTICS — Successful Trials Only]
  Mean speedup   : 1.028x
  Median speedup : 1.003x
  Best speedup   : 1.143x  (task: 16_Matmul_with_transposed_A)
  Worst speedup  : 0.998x  (task: 9_Tall_skinny_matrix_multiplication_)
  Kernels faster than baseline  : 5/6 (83%)
  Kernels slower than baseline  : 1/6 (17%)

[LLM PERFORMANCE]
  Mean generation time   : 15.0s
  Total generation time  : 300.5s

[FULL TRIAL TABLE]
    trial_id                               task_name            status    reward  llm_time_s occupancy_pct compute_pct memory_pct
baseline_000         1_Square_matrix_multiplication_ COMPILATION_ERROR -1.000000       23.17          None        N